 # Whisper Accent Robustness — Model Performance Evaluation



 Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.



 - **Scripted**     → 6 held-out test speakers (never seen during training)

 - **Spontaneous**  → suitcase corpus (OOD, all speakers)



 Metrics:

 - **WER**  — word error rate (primary ASR metric)

 - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [26]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"

# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline":      "Zero-shot",
    "baseline_lora": "Naive LoRA FT",
    "no_aux":        "No Aux",
    "ctc_aux":       "CTC Aux",
    "feat_aux":      "Feat Aux",
    "both_aux":      "CTC + Feat",
}
SPLITS = ["scripted", "spontaneous"]


In [27]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from jiwer import wer as jiwer_wer

pd.set_option("display.max_colwidth", 80)


In [28]:
def load_results(model_key: str, split: str) -> pd.DataFrame | None:
    p = Path(RESULTS_DIR) / f"{model_key}_{split}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)

# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, dict[str, pd.DataFrame | None]] = {}
for key in MODEL_KEYS:
    results[key] = {}
    for split in SPLITS:
        df = load_results(key, split)
        results[key][split] = df
        if df is not None:
            print(f"  loaded  {key}/{split}: {len(df):,} rows")


  loaded  baseline/scripted: 6,506 rows
  loaded  baseline/spontaneous: 22 rows
  loaded  baseline_lora/scripted: 6,506 rows
  loaded  baseline_lora/spontaneous: 22 rows
  loaded  no_aux/scripted: 6,506 rows
  loaded  no_aux/spontaneous: 22 rows
  loaded  ctc_aux/scripted: 6,506 rows
  loaded  ctc_aux/spontaneous: 22 rows
  [missing] results/model_perf_comparison/feat_aux_scripted_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/feat_aux_spontaneous_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/both_aux_scripted_predictions.csv — run eval_model_perf.py first
  [missing] results/model_perf_comparison/both_aux_spontaneous_predictions.csv — run eval_model_perf.py first


In [29]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def available(key: str, split: str) -> bool:
    return results.get(key, {}).get(split) is not None


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer_wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))


def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived), precomputed by eval_model_perf.py."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer_wer(
                            sub["reference_norm"].fillna("").tolist(),
                            sub["prediction_norm"].fillna("").tolist(),
                        )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")


Helpers loaded.


 ---

 # Part 1 — Overall WER & PER (G2P)

In [30]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for key, label in MODEL_KEYS.items():
    for split in SPLITS:
        if not available(key, split):
            continue
        df  = results[key][split]
        wer = corpus_wer(df)
        per = corpus_per(df)
        rows.append({"Model": label, "Split": split, "WER": wer, "PER (G2P)": per})

overall_df = pd.DataFrame(rows)

for metric in ["WER", "PER (G2P)"]:
    pivot = overall_df.pivot(index="Model", columns="Split", values=metric)
    display(
        pivot.style
             .format("{:.4f}")
             .background_gradient(cmap="RdYlGn_r", axis=None)
             .set_caption(f"{metric} by Model × Split (lower is better)")
    )


Split,scripted,spontaneous
Model,,
CTC Aux,1.5115,0.6382
Naive LoRA FT,0.0889,0.6261
No Aux,4.0056,0.6359
Zero-shot,0.1841,0.6369


Split,scripted,spontaneous
Model,,
CTC Aux,1.7008,0.5412
Naive LoRA FT,0.0549,0.5366
No Aux,4.8817,0.5393
Zero-shot,0.1004,0.5448


In [31]:
# ── Bar charts: WER and PER side-by-side, one figure per split ───────────────
for split in SPLITS:
    sub = overall_df[overall_df["Split"] == split].dropna(subset=["WER"])
    if sub.empty:
        continue

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name         = "WER",
        x            = sub["Model"].tolist(),
        y            = sub["WER"].tolist(),
        text         = [f"{v:.1%}" for v in sub["WER"]],
        textposition = "outside",
    ))
    if sub["PER (G2P)"].notna().any():
        fig.add_trace(go.Bar(
            name         = "PER (G2P)",
            x            = sub["Model"].tolist(),
            y            = sub["PER (G2P)"].tolist(),
            text         = [f"{v:.1%}" if pd.notna(v) else "" for v in sub["PER (G2P)"]],
            textposition = "outside",
        ))
    fig.update_layout(
        title   = f"WER & PER by Model — {split.capitalize()}",
        barmode = "group",
        yaxis   = dict(title="Error Rate", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin  = dict(t=80),
    )
    fig.show()


 ---

 # Part 2 — Per-L1 WER

In [32]:
for split in SPLITS:
    l1_rows = []
    for key, label in MODEL_KEYS.items():
        if not available(key, split):
            continue
        grp = grouped_wer(results[key][split], "l1")
        grp["Model"] = label
        l1_rows.append(grp)
    if not l1_rows:
        continue

    l1_df = pd.concat(l1_rows, ignore_index=True)

    # Delta vs zero-shot baseline (negative = improvement)
    base_label = MODEL_KEYS["baseline"]
    base_wer   = (
        l1_df[l1_df["Model"] == base_label][["l1", "wer"]]
        .rename(columns={"wer": "wer_base"})
    )
    l1_df = l1_df.merge(base_wer, on="l1", how="left")
    l1_df["wer_delta_pct"] = (
        (l1_df["wer"] - l1_df["wer_base"]) / l1_df["wer_base"] * 100
    )

    l1s = sorted(l1_df["l1"].unique())

    fig = go.Figure()
    for key, label in MODEL_KEYS.items():
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name = label,
            x    = l1s,
            y    = [sub.loc[l, "wer"] if l in sub.index else None for l in l1s],
            text = [f"{sub.loc[l, 'wer']:.1%}" if l in sub.index else "" for l in l1s],
            textposition = "outside",
        ))
    fig.update_layout(
        title   = f"WER by L1 — {split.capitalize()}",
        barmode = "group",
        yaxis   = dict(title="WER", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin  = dict(t=80),
    )
    fig.show()

    out = Path(RESULTS_DIR) / f"comparison_{split}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")
    display(
        l1_df.sort_values(["l1", "Model"])
             .style.format({
                 "wer":           "{:.4f}",
                 "wer_base":      "{:.4f}",
                 "wer_delta_pct": "{:+.1f}%",
                 "per":           lambda v: f"{v:.4f}" if pd.notna(v) else "—",
             })
             .set_caption(f"{split.capitalize()} — Per-L1 WER")
    )


Saved results/model_perf_comparison/comparison_scripted_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
18,Arabic,974,1.7636,1.9503,CTC Aux,0.2387,+638.9%
6,Arabic,974,0.1125,0.0735,Naive LoRA FT,0.2387,-52.9%
12,Arabic,974,2.6352,3.3911,No Aux,0.2387,+1004.1%
0,Arabic,974,0.2387,0.1374,Zero-shot,0.2387,+0.0%
19,Chinese,1130,1.2769,1.5497,CTC Aux,0.1916,+566.6%
7,Chinese,1130,0.0987,0.0582,Naive LoRA FT,0.1916,-48.5%
13,Chinese,1130,4.2583,5.2404,No Aux,0.1916,+2122.9%
1,Chinese,1130,0.1916,0.1055,Zero-shot,0.1916,+0.0%
20,Hindi,1132,1.9034,2.0982,CTC Aux,0.0653,+2813.5%
8,Hindi,1132,0.0094,0.0051,Naive LoRA FT,0.0653,-85.5%


Saved results/model_perf_comparison/comparison_spontaneous_by_l1.csv


,l1,num_utts,wer,per,Model,wer_base,wer_delta_pct
18,Arabic,3,0.5389,0.5032,CTC Aux,0.5337,+1.0%
6,Arabic,3,0.5259,0.4985,Naive LoRA FT,0.5337,-1.5%
12,Arabic,3,0.5389,0.4997,No Aux,0.5337,+1.0%
0,Arabic,3,0.5337,0.5104,Zero-shot,0.5337,+0.0%
19,Chinese,4,0.6314,0.5916,CTC Aux,0.6423,-1.7%
7,Chinese,4,0.6241,0.5899,Naive LoRA FT,0.6423,-2.8%
13,Chinese,4,0.6369,0.5941,No Aux,0.6423,-0.9%
1,Chinese,4,0.6423,0.6078,Zero-shot,0.6423,+0.0%
20,Hindi,3,0.4663,0.3724,CTC Aux,0.4340,+7.4%
8,Hindi,3,0.4487,0.3621,Naive LoRA FT,0.4340,+3.4%


 ---

 # Part 3 — Scripted vs Spontaneous Gap (zero-shot)



 Different corpora → compared at L1 level, not speaker level.

In [33]:
if available("baseline", "scripted") and available("baseline", "spontaneous"):
    s_g  = grouped_wer(results["baseline"]["scripted"],    "l1").rename(columns={"wer": "WER_scripted"})
    sp_g = grouped_wer(results["baseline"]["spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    gap  = s_g[["l1", "WER_scripted"]].merge(
               sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
    gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]

    l1s = gap["l1"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Scripted",    x=l1s, y=gap["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
    fig.update_layout(
        title   = "Zero-shot WER — Scripted vs Spontaneous by L1",
        barmode = "group",
        yaxis   = dict(title="WER", tickformat=".0%"),
        legend  = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()
    display(
        gap.style
           .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
           .set_caption("Scripted vs Spontaneous WER gap (zero-shot)")
    )


,l1,WER_scripted,WER_spontaneous,gap
0,Arabic,0.2387,0.5337,0.2950
1,Chinese,0.1916,0.6423,0.4508
2,Hindi,0.0653,0.4340,0.3687
3,Korean,0.0810,0.8089,0.7279
4,Spanish,0.2284,0.5957,0.3674
5,Vietnamese,0.3123,0.5510,0.2387


 ---

 # Part 4 — Utterance-level Analysis

In [34]:
# ── UTT WER distributions ─────────────────────────────────────────────────────
fig = go.Figure()
key = "ctc_aux"
label = MODEL_KEYS[key]
split = "scripted"
fig.add_trace(go.Histogram(
    x       = results[key][split]["utt_wer"],
    name    = f"{label} / {split}",
    opacity = 0.5,
    nbinsx  = 40,
))
fig.update_layout(
    title    = "Utterance WER Distribution — All Conditions",
    barmode  = "overlay",
    xaxis    = dict(title="Utterance WER"),
    yaxis    = dict(title="Count"),
    legend   = dict(orientation="h", y=1.12, xanchor="center", x=0.5),
)
fig.show()


In [35]:
# ── Worst utterances per model — scripted, cross-model comparison ─────────────
split    = "scripted"
N_WORST  = 15

base_df = results["baseline"][split]

for anchor_key, anchor_label in MODEL_KEYS.items():
    if not available(anchor_key, split):
        continue

    anchor_df = results[anchor_key][split]
    idx       = anchor_df.nlargest(N_WORST, "utt_wer").index

    worst = base_df.loc[idx, ["utterance_id", "speaker", "l1", "reference_norm"]].copy()

    # Add each model's prediction + WER + PER for these utterances
    for key, label in MODEL_KEYS.items():
        if not available(key, split):
            continue
        other            = results[key][split]
        col              = label.replace(" ", "_").lower()
        worst[f"pred_{col}"] = other.loc[idx, "prediction_norm"].values
        worst[f"wer_{col}"]  = other.loc[idx, "utt_wer"].values
        if "utt_per" in other.columns:
            worst[f"per_{col}"] = other.loc[idx, "utt_per"].values

    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(("wer_", "per_"))}
    display(
        worst.style
                .format(fmt_cols)
                .set_caption(f"Top-{N_WORST} Worst Utterances for {anchor_label} — Scripted")
    )

,utterance_id,speaker,l1,reference_norm,pred_zero-shot,wer_zero-shot,per_zero-shot,pred_naive_lora_ft,wer_naive_lora_ft,per_naive_lora_ft,pred_no_aux,wer_no_aux,per_no_aux,pred_ctc_aux,wer_ctc_aux,per_ctc_aux
3422,arctic_a0155,HQTV,Vietnamese,wont you draw up gentlemen,one youve rarred and the other youve chandlemen,1.600,1.050,wont you draw up gentlemen,0.000,0.000,wont you draw up gentlemen,0.000,0.000,wont you draw up gentlemen,0.000,0.000
4179,arctic_b0319,HQTV,Vietnamese,daylight was tired profoundly tired,they lied to a tire a foully tire,1.600,0.480,they liked what tide roughly tide,1.200,0.440,their life was tired profoundly tired,0.400,0.160,daylight was tired the profile tired the profile tired the profile tired the profile,2.000,1.360
1510,arctic_a0381,EBVS,Spanish,my names ferguson,my name is swargu sam,1.333,0.615,my name is swargoon,1.000,0.462,my names ferguson,0.000,0.000,my names ferguson,0.000,0.000
4780,arctic_a0381,SKA,Arabic,my names ferguson,my name is faye gasson,1.333,0.231,my name is gregson,1.000,0.385,my names ferguson,0.000,0.000,my names ferguson,0.000,0.000
3577,arctic_a0310,HQTV,Vietnamese,massage under tension was the cryptic reply,much of the charges and the tensions were a critical reply,1.286,0.625,much much charge and attention was a great reply,1.000,0.438,much chargers and tansons were cryptic replies the more he replied,1.429,0.844,much chargers and tensions were a great reply,1.000,0.531
4188,arctic_b0328,HQTV,Vietnamese,change chairs daylight commanded,change chances they like command it,1.250,0.381,change chairs they like commanded,0.500,0.095,change chairs daylight commanded the man to a fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire fire,109.500,62.381,change chairs they like commanded,0.500,0.095
1364,arctic_a0235,EBVS,Spanish,his voice was passionately rebellious,his voice was fascinating and it was a reb

,utterance_id,speaker,l1,reference_norm,pred_zero-shot,wer_zero-shot,per_zero-shot,pred_naive_lora_ft,wer_naive_lora_ft,per_naive_lora_ft,pred_no_aux,wer_no_aux,per_no_aux,pred_ctc_aux,wer_ctc_aux,per_ctc_aux
4561,arctic_a0162,SKA,Arabic,thats the sub foreman explained thorpe,thats the soft form in except playing this rope,1.167,0.481,that is the self for man except playing the thrope,1.500,0.593,thats the sub foreman explaining the throepad,0.500,0.296,that is the soft foreman except in the strop of a nail,1.667,0.741
3282,arctic_a0015,HQTV,Vietnamese,its the aurora borealis,is it already boring,1.000,0.824,its an old rock bore alice,1.250,0.471,hes an aurora borealis,0.500,0.294,he saw all the raw bones of the house,2.000,0.941
4179,arctic_b0319,HQTV,Vietnamese,daylight was tired profoundly tired,they lied to a tire a foully tire,1.600,0.480,they liked what tide roughly tide,1.200,0.440,their life was tired profoundly tired,0.400,0.160,daylight was tired the profile tired the profile tired the profile tired the profile,2.000,1.360
5342,arctic_b0351,SKA,Arabic,matthewson whos this bookkeeper rogers,make some holes with the bookkeeper,1.200,0.615,matt on who this book keeper regards,1.200,0.385,met on horses bookkeeper rogersed,0.800,0.654,matthew and horaces bookkeeper rogers,0.600,0.308
4128,arctic_b0268,HQTV,Vietnamese,saxon nodded and the boy frowned,susan noted and the boy frowned,0.333,0.217,such a note is in the boys throne,1.167,0.609,saxon nodded and the boy frowned,0.000,0.000,such a noise did the boy fronts be heard in the morning,1.667,1.130
5143,arctic_b0152,SKA,Arabic,ow a wild dog he growled,i have a very big head,1.000,0.875,ah how how how he looked he cried,1.167,1.000,ah ah ah ah ah he cried,1.000,0.750,i heard lopis look he cried,0.833,0.875
14,arctic_a0015,BWC,Chinese,its the aurora borealis,is a rural place,1.000,0.647,its a roll up brilliant,1.000,0.647,its the roll of brilliance,0.750,0.529,its a roll out brilliant,1.000,0.706
380,arctic_a0381,BWC,Chinese,my names ferguson,my name is fergusson,1.000,0.077,my name is fergusson,1.000,0.077,my names fergusson and im the father of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of fergusson and im the son of ferg and im the son of ferg and im the son of ferg and im the son of ferg and im the son of ferg and im the son of ferg and im the son of ferg and im the son of ferg and im,91.000,63.769,my names ferrault kussin,0.667,0.462
1392,arctic_a0263,EBVS,Spanish,joan looked triumphantly at sheldon who bowed,join lookitrumpfully and shut down cooperate,1.000,0.677,joan looked at him trembling and shat down cooing,1.000,0.677,joan looked triumphantly and shied down coolly and said,0.857,0.452,joan looked trembling and shuddered coaxed him coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coaxing coax

,utterance_id,speaker,l1,reference_norm,pred_zero-shot,wer_zero-shot,per_zero-shot,pred_naive_lora_ft,wer_naive_lora_ft,per_naive_lora_ft,pred_no_aux,wer_no_aux,per_no_aux,pred_ctc_aux,wer_ctc_aux,per_ctc_aux
328,arctic_a0329,BWC,Chinese,ah indeed,aha indeed,0.500,0.333,ah indeed,0.000,0.000,ah indeed i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know i know,221.000,110.500,ah indeed,0.000,0.000
1458,arctic_a0329,EBVS,Spanish,ah indeed,are indeed,0.500,0.167,ah indeed,0.000,0.000,ah indeed i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say i say,221.000,110.500,ah indeed,0.000,0.000
5702,arctic_a0329,SVBI,Hindi,ah indeed,ah indeed,0.000,0.000,ah indeed,0.000,0.000,ah indeed i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said i said

,utterance_id,speaker,l1,reference_norm,pred_zero-shot,wer_zero-shot,per_zero-shot,pred_naive_lora_ft,wer_naive_lora_ft,per_naive_lora_ft,pred_no_aux,wer_no_aux,per_no_aux,pred_ctc_aux,wer_ctc_aux,per_ctc_aux
1887,arctic_b0166,EBVS,Spanish,fast but endure,fast but endure,0.000,0.000,fast but endure,0.000,0.000,fast but endure a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit more a little bit a,146.667,101.385,fast but endure it to be true to love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love love l

In [36]:
# ── Utterance-level head-to-head win counts ───────────────────────────────────
split = "scripted"   # change to "spontaneous" if needed

available_keys = [k for k in MODEL_KEYS if available(k, split)]

for anchor_key in available_keys:
    anchor_label = MODEL_KEYS[anchor_key]
    anchor_wer   = results[anchor_key][split]["utt_wer"].fillna(1.0)
    n_utts       = len(anchor_wer)

    print(f"\n{anchor_label} vs others ({split}, n={n_utts} utterances):")
    for other_key in available_keys:
        if other_key == anchor_key:
            continue
        other_wer  = results[other_key][split]["utt_wer"].fillna(1.0)
        anchor_label_short = MODEL_KEYS[other_key]

        wins  = (anchor_wer < other_wer).sum()
        ties  = (anchor_wer == other_wer).sum()
        loses = (anchor_wer > other_wer).sum()

        print(f"  vs {anchor_label_short:<20} | "
              f"wins={wins:>4} ({wins/n_utts:.1%})  "
              f"ties={ties:>4} ({ties/n_utts:.1%})  "
              f"loses={loses:>4} ({loses/n_utts:.1%})")


Zero-shot vs others (scripted, n=6506 utterances):
  vs Naive LoRA FT        | wins= 362 (5.6%)  ties=3043 (46.8%)  loses=3101 (47.7%)
  vs No Aux               | wins=1437 (22.1%)  ties=2214 (34.0%)  loses=2855 (43.9%)
  vs CTC Aux              | wins=1026 (15.8%)  ties=2412 (37.1%)  loses=3068 (47.2%)

Naive LoRA FT vs others (scripted, n=6506 utterances):
  vs Zero-shot            | wins=3101 (47.7%)  ties=3043 (46.8%)  loses= 362 (5.6%)
  vs No Aux               | wins=1692 (26.0%)  ties=3829 (58.9%)  loses= 985 (15.1%)
  vs CTC Aux              | wins=1346 (20.7%)  ties=4011 (61.7%)  loses=1149 (17.7%)

No Aux vs others (scripted, n=6506 utterances):
  vs Zero-shot            | wins=2855 (43.9%)  ties=2214 (34.0%)  loses=1437 (22.1%)
  vs Naive LoRA FT        | wins= 985 (15.1%)  ties=3829 (58.9%)  loses=1692 (26.0%)
  vs CTC Aux              | wins=1149 (17.7%)  ties=3659 (56.2%)  loses=1698 (26.1%)

CTC Aux vs others (scripted, n=6506 utterances):
  vs Zero-shot            | wi

# Held-out L1 Performance - Accent-Robustness

In [37]:
# ── Held-out L1 comparison across selected models ─────────────────────────────
import pandas as pd
from IPython.display import display

held_out_l1 = "Chinese"   # change this
split = "scripted"         # or "spontaneous"
model_keys = [
    "baseline",
    "baseline_lora_ho_chinese",
    "ctc_aux_ho_chinese",
]

rows = []
for key in model_keys:
    if not available(key, split):
        print(f"[WARN] missing: {key} / {split}")
        continue

    df = results[key][split].copy()
    df = df[df["l1"] == held_out_l1]

    if len(df) == 0:
        print(f"[WARN] no rows for {key} on L1={held_out_l1}")
        continue

    row = {
        "model_key": key,
        "model": MODEL_KEYS.get(key, key),
        "n_utts": len(df),
        "n_speakers": df["speaker"].nunique() if "speaker" in df.columns else None,
        "wer_mean": df["utt_wer"].mean(),
        "wer_std": df["utt_wer"].std(),
        "wer_median": df["utt_wer"].median(),
    }

    if "utt_per" in df.columns:
        row["per_mean"] = df["utt_per"].mean()
        row["per_std"] = df["utt_per"].std()
        row["per_median"] = df["utt_per"].median()

    rows.append(row)

heldout_cmp = pd.DataFrame(rows).sort_values("wer_mean").reset_index(drop=True)

fmt = {
    c: "{:.3f}"
    for c in heldout_cmp.columns
    if c.startswith(("wer_", "per_"))
}

display(
    heldout_cmp.style
        .format(fmt)
        .set_caption(f"Held-out L1 comparison — {held_out_l1} / {split}")
)

[WARN] missing: baseline_lora_ho_chinese / scripted
[WARN] missing: ctc_aux_ho_chinese / scripted


,model_key,model,n_utts,n_speakers,wer_mean,wer_std,wer_median,per_mean,per_std,per_median
0,baseline,Zero-shot,1130,1,0.200,0.206,0.154,0.105,0.114,0.080
